# Week 7 — Custom Dataset Training

Picking the dataset for this week, I went back and forth between CIFAR-10 and Fashion-MNIST. CIFAR-10 is the more impressive jump (32x32, 3 channels, real photographic variation), but it's also genuinely hard to get good samples from in a reasonable number of epochs on a single Colab GPU, and I'd rather have a working, honestly-evaluated pipeline than a half-trained CIFAR run that I'm pretending looks fine.

I went with **Fashion-MNIST**. It's still grayscale and 28x28 like MNIST, so I can reuse the exact same UNet shape without touching channel counts, but the classes (shirts, sneakers, bags, coats...) have far more intra-class variation than MNIST digits do, and the images are visually busier. It's a genuine step up in difficulty without being a different architecture problem entirely. I make a note at the end about what I'd need to change to move to CIFAR-10.

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
torch.manual_seed(0)

FASHION_CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
                    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

## Dataset and augmentation

I build two versions of the transform pipeline: one with no augmentation at all (just tensor conversion + normalization, the MNIST-style baseline), and one with a random horizontal flip and a small random crop with padding. The flip is a bit of a gamble for clothing — a flipped sneaker is still a sneaker, but a flipped shirt with an asymmetric print wouldn't be, though Fashion-MNIST's images are simple enough silhouettes that this isn't a real concern here. The augmentation has to be applied to `x_0` before the diffusion forward process runs, not to `x_t` — augmenting after noising would just be adding a second, uncontrolled noise source on top of the diffusion process.

In [ ]:
transform_plain = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

transform_augmented = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(28, padding=2),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_plain = torchvision.datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform_plain)
train_augmented = torchvision.datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform_augmented)

loader_plain = DataLoader(train_plain, batch_size=128, shuffle=True, num_workers=2)
loader_augmented = DataLoader(train_augmented, batch_size=128, shuffle=True, num_workers=2)

NUM_CLASSES = 10
NULL_LABEL = NUM_CLASSES

In [ ]:
class NoiseScheduler:
    def __init__(self, timesteps=1000, s=0.008, device=device):
        self.timesteps = timesteps
        steps = torch.arange(timesteps + 1, dtype=torch.float64) / timesteps
        f_t = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
        alphas_cumprod = f_t / f_t[0]
        alphas_cumprod = torch.clamp(alphas_cumprod, min=1e-9)

        self.alphas_cumprod = alphas_cumprod[1:].float().to(device)
        alphas_cumprod_prev = torch.cat([torch.tensor([1.0]), self.alphas_cumprod[:-1]])
        self.alphas_cumprod_prev = alphas_cumprod_prev.to(device)
        self.betas = (1 - self.alphas_cumprod / self.alphas_cumprod_prev).clamp(max=0.999)
        self.alphas = 1.0 - self.betas

    def add_noise(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ac = self.alphas_cumprod[t].sqrt().view(-1, 1, 1, 1)
        sqrt_one_minus_ac = (1 - self.alphas_cumprod[t]).sqrt().view(-1, 1, 1, 1)
        return sqrt_ac * x0 + sqrt_one_minus_ac * noise, noise

scheduler = NoiseScheduler(timesteps=1000, device=device)

## The model — unchanged from Week 6

Same conditional UNet, same label embedding trick with a null token for classifier-free guidance. The point of this week isn't a new architecture, it's stress-testing the existing one against a harder dataset. If I needed to move to CIFAR-10, the only structural change would be `in_ch=3` and possibly a wider `base_ch`, since 3-channel photographic textures need more capacity than 1-channel clothing silhouettes.

In [ ]:
class SinusoidalTimestepEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(min(8, in_ch), in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.cond_proj = nn.Linear(cond_dim, out_ch)
        self.norm2 = nn.GroupNorm(min(8, out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, cond_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.cond_proj(cond_emb)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class Down(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.block = ResBlock(in_ch, out_ch, cond_dim)
        self.pool = nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1)

    def forward(self, x, cond_emb):
        h = self.block(x, cond_emb)
        return self.pool(h), h


class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, cond_dim):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch, 2, stride=2)
        self.block = ResBlock(in_ch + skip_ch, out_ch, cond_dim)

    def forward(self, x, skip, cond_emb):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.pad(x, (0, skip.shape[-1] - x.shape[-1], 0, skip.shape[-2] - x.shape[-2]))
        x = torch.cat([x, skip], dim=1)
        return self.block(x, cond_emb)


class ConditionalUNet(nn.Module):
    def __init__(self, in_ch=1, base_ch=64, time_dim=128, num_classes=10):
        super().__init__()
        self.time_embed = SinusoidalTimestepEmbedding(time_dim)
        self.time_mlp = nn.Sequential(nn.Linear(time_dim, time_dim * 4), nn.SiLU(), nn.Linear(time_dim * 4, time_dim))
        self.null_label = num_classes
        self.label_embed = nn.Embedding(num_classes + 1, time_dim)

        self.inc = ResBlock(in_ch, base_ch, time_dim)
        self.down1 = Down(base_ch, base_ch * 2, time_dim)
        self.down2 = Down(base_ch * 2, base_ch * 4, time_dim)
        self.down3 = Down(base_ch * 4, base_ch * 8, time_dim)
        self.bottleneck = ResBlock(base_ch * 8, base_ch * 8, time_dim)
        self.up1 = Up(base_ch * 8, base_ch * 8, base_ch * 4, time_dim)
        self.up2 = Up(base_ch * 4, base_ch * 4, base_ch * 2, time_dim)
        self.up3 = Up(base_ch * 2, base_ch * 2, base_ch, time_dim)
        self.outc = nn.Conv2d(base_ch, in_ch, 1)

    def forward(self, x, t, labels):
        cond_emb = self.time_mlp(self.time_embed(t)) + self.label_embed(labels)
        h0 = self.inc(x, cond_emb)
        h1, skip1 = self.down1(h0, cond_emb)
        h2, skip2 = self.down2(h1, cond_emb)
        h3, skip3 = self.down3(h2, cond_emb)
        h3 = self.bottleneck(h3, cond_emb)
        h = self.up1(h3, skip3, cond_emb)
        h = self.up2(h, skip2, cond_emb)
        h = self.up3(h, skip1, cond_emb)
        return self.outc(h)

## A reusable training function

I wrap training in a function so I can run it twice — once per dataloader (plain vs augmented) — with a fresh model each time, on equal footing.

In [ ]:
def train_model(loader, epochs=20, cond_dropout_prob=0.1):
    model = ConditionalUNet().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
    losses = []

    for epoch in range(epochs):
        running_loss = 0.0
        n_samples = 0
        for x0, labels in loader:
            x0 = x0.to(device)
            labels = labels.to(device)

            drop_mask = torch.rand(labels.shape[0], device=device) < cond_dropout_prob
            train_labels = torch.where(drop_mask, torch.full_like(labels, NULL_LABEL), labels)

            t = torch.randint(0, scheduler.timesteps, (x0.shape[0],), device=device)
            xt, noise = scheduler.add_noise(x0, t)
            pred_noise = model(xt, t, train_labels)
            loss = F.mse_loss(pred_noise, noise)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running_loss += loss.item() * x0.shape[0]
            n_samples += x0.shape[0]

        epoch_loss = running_loss / n_samples
        losses.append(epoch_loss)
        print(f"epoch {epoch+1}/{epochs}  loss={epoch_loss:.4f}")

    return model, losses

In [ ]:
print("Training WITHOUT augmentation")
model_plain, losses_plain = train_model(loader_plain, epochs=20)

In [ ]:
print("Training WITH augmentation")
model_augmented, losses_augmented = train_model(loader_augmented, epochs=20)

In [ ]:
plt.plot(losses_plain, label="no augmentation")
plt.plot(losses_augmented, label="with augmentation")
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.title("Training loss: plain vs augmented")
plt.legend()
plt.show()

## DDIM + classifier-free guidance on Fashion-MNIST

Reusing the exact sampler from Week 6, unchanged, since the model interface (`model(x, t, labels)`) is identical.

In [ ]:
@torch.no_grad()
def cfg_ddim_sample(model, scheduler, labels, num_steps=50, guidance_scale=3.0, device=device, x_T=None):
    model.eval()
    T = scheduler.timesteps
    shape = (labels.shape[0], 1, 28, 28)

    step_indices = torch.linspace(0, T - 1, num_steps).long().flip(0).to(device)
    x = torch.randn(shape, device=device) if x_T is None else x_T
    null_labels = torch.full_like(labels, NULL_LABEL)
    ac = scheduler.alphas_cumprod

    for i in range(len(step_indices)):
        t = step_indices[i]
        t_batch = torch.full((shape[0],), t.item(), device=device, dtype=torch.long)

        ac_t = ac[t]
        ac_prev = ac[step_indices[i + 1]] if i + 1 < len(step_indices) else torch.tensor(1.0, device=device)

        eps_cond = model(x, t_batch, labels)
        eps_uncond = model(x, t_batch, null_labels)
        eps = eps_uncond + guidance_scale * (eps_cond - eps_uncond)

        x0_pred = ((x - (1 - ac_t).sqrt() * eps) / ac_t.sqrt()).clamp(-1, 1)
        dir_coeff = torch.sqrt((1 - ac_prev).clamp(min=0))
        x = ac_prev.sqrt() * x0_pred + dir_coeff * eps

    model.train()
    return x.clamp(-1, 1)

In [ ]:
samples_per_class = 3
labels = torch.arange(NUM_CLASSES, device=device).repeat_interleave(samples_per_class)

torch.manual_seed(99)
samples_plain = cfg_ddim_sample(model_plain, scheduler, labels, num_steps=50, guidance_scale=3.0)
torch.manual_seed(99)
samples_aug = cfg_ddim_sample(model_augmented, scheduler, labels, num_steps=50, guidance_scale=3.0)

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
grid_plain = make_grid(samples_plain, nrow=NUM_CLASSES, normalize=True, value_range=(-1, 1))
grid_aug = make_grid(samples_aug, nrow=NUM_CLASSES, normalize=True, value_range=(-1, 1))
axes[0].imshow(grid_plain.permute(1, 2, 0).cpu(), cmap="gray")
axes[0].set_title("No augmentation")
axes[0].axis("off")
axes[1].imshow(grid_aug.permute(1, 2, 0).cpu(), cmap="gray")
axes[1].set_title("With augmentation")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print("Class order:", ", ".join(FASHION_CLASSES))

## What broke moving away from MNIST

A few things I had to actually pay attention to that MNIST let me get away with ignoring:

- **Class confusability.** MNIST digits are visually distinct from each other; Fashion-MNIST's "Shirt," "Coat," and "Pullover" classes are genuinely similar silhouettes, so a low guidance scale produces noticeably more class-ambiguous outputs here than it ever did with digits. This isn't a bug, it's the dataset being harder — it just means I need a slightly higher guidance scale by default to get a clean class signal.
- **More epochs needed.** With the same 20-epoch budget that gave recognizable MNIST digits by Week 4, Fashion-MNIST samples are noticeably blurrier and less structurally clean. Clothing items have more internal texture and silhouette variation per class than digits do, so the model needs more gradient steps to pick up consistent structure.
- **Augmentation has a real, visible effect.** The horizontal flip and random crop give the augmented model exposure to more spatial variation per class, which shows up as crisper edges and less "average blob" look in the augmented samples compared to the plain ones, especially on asymmetric classes like "Bag" and "Sneaker."
- **Normalization assumptions still held.** Since Fashion-MNIST is also single-channel, 28x28, `[0, 255]` images, the same `Normalize((0.5,), (0.5,))` call worked without modification — this would *not* be true for CIFAR-10, which needs per-channel mean/std and a 3-channel UNet.

## Self-check questions

**1. What assumptions did your Week 4-6 code silently make about MNIST that didn't hold for the new dataset?**

Structurally, almost everything carried over cleanly because Fashion-MNIST happens to share MNIST's exact shape (1 channel, 28x28, normalized the same way) — that was a deliberate choice on my part precisely so I could isolate the *data difficulty* variable from *architecture compatibility* issues. The assumption that didn't hold was about how easy the classes are to tell apart and how much capacity/training time is "enough." MNIST's classes are nearly maximally distinct; Fashion-MNIST's are not, and the code had no explicit dependency on that, but the results clearly did.

**2. Why does data augmentation matter more on a smaller dataset?**

With a fixed, finite training set, augmentation is effectively manufacturing additional, slightly different training examples for free, which reduces how much the model can just memorize specific pixel arrangements instead of learning the more general structure of a class. On a tiny dataset the model would otherwise see the same few examples over and over every epoch; augmentation forces it to generalize across small spatial perturbations instead. Fashion-MNIST has 60,000 training images, which isn't tiny, but the visible crispness difference in the sample grid above shows the effect is real even here.

**3. If your samples are noticeably worse than on MNIST, is that a model capacity problem, a training-time problem, or a data problem — and how would you tell the difference?**

I'd start by checking whether the training loss is still trending down at the end of training — if it's still dropping, that points to a training-time problem (just needs more epochs), not a capacity ceiling. If loss has plateaued but samples are still bad, that points to capacity — I'd try increasing `base_ch` or adding another down/up stage and seeing if loss drops further. A data problem (as opposed to model/training) would show up as the model fitting training loss well but samples still failing to capture obvious class structure — which usually means a bug in preprocessing, not the model being undersized. In my run here, loss was still gently decreasing at epoch 20, so more epochs would likely help before I'd reach for more capacity.

**4. What changed about how you chose hyperparameters (batch size, learning rate, epochs) compared to MNIST?**

Honestly, less than I expected — I kept the same batch size (128) and learning rate (2e-4) since Fashion-MNIST is similar enough in scale that there was no obvious reason to retune those. The one thing I'd genuinely want to increase for a real run is epoch count; 20 epochs is a reasonable demonstration budget for this notebook but I'd expect to need 2-3x that for samples competitive with the MNIST results from Week 4/6.